In [ ]:
import polyase as poly
import pandas as pd
import RNApysoforms as RNApy
import polars as pl

In [ ]:
poly.__version__

# Running Polyase on tetraploid potato sample 

### Required Files



- **Syntelog finder output**: To assign genes based on syntelogous relationships
- **Oarfish directory**: Contains transcript quantification data
- **GFF file (liftoff annotation)**: Genome annotation from liftoff
- **GFF file (novel transcripts)**: Novel identified transcripts
- **Functional gene annotation**: Gene functional annotations

### Installation of polyase 


You can install polyase using pip:
```bash
pip install polyase
```


## 1) Reading in the data 

In this section we will read in the data genespace output file, transcript quanitfication directory, tx2gene. 

In [ ]:


# Metadata for genes
var_obs_file = "potato-allelic-orthogroups/output_desiree_liftoff/genespace/De_v1.unitato_liftoff_haplotap_genespace_categories.tsv"
# directory of gene counts from oarfish
gene_counts_dir = "nf-core-plantlongrnaseq/output_desiree_liftoff_phased/oarfish/"
tx2gene = "nf-core-plantlongrnaseq/output_desiree_liftoff_phased/custom/combined.tx2gene.tsv"
#nf-core-plantlongrnaseq/work/3f/6abd9f23b65ae2de98213bcc0ce3a5
# Define sample IDs and their conditions
sample_info = {
    "DE_C_H_13": "control",
    "DE_C_H_14": "control",
    "DE_C_H_15": "control",
    "DE_C_H_16": "control",
    "DE_H_H_17": "heat",
    "DE_H_H_18": "heat",
    "DE_H_H_19": "heat",
    "DE_H_H_20": "heat"
} 

### Dataset Information



In [ ]:


# With isoforms
adata_isoform = poly.load_ase_data(
    var_obs_file=var_obs_file,
    sample_info=sample_info,
    tx_to_gene_file=tx2gene,
    isoform_counts_dir=gene_counts_dir,
    quant_dir=gene_counts_dir
)

## Update with batch corrected counts

In [ ]:
em_batch_corrected_counts = pd.read_csv("nf-core-plantlongrnaseq/output_desiree_liftoff_phased/tximeta/all_samples.transcript_counts_combat_seq_corrected.tsv", sep="\t", index_col=0)
em_batch_corrected_counts.drop(columns=['gene_id'], inplace=True)
unique_counts_batch_corrected = pd.read_csv("nf-core-plantlongrnaseq/output_desiree_liftoff_phased/oarfish_batch_corrected/allelic_unique_transcript_counts.tsv", sep="\t", index_col=0)
ambiguous_counts_batch_corrected = pd.read_csv("nf-core-plantlongrnaseq/output_desiree_liftoff_phased/oarfish_batch_corrected/allelic_ambiguous_transcript_counts.tsv", sep="\t", index_col=0)
# rename index to tx
unique_counts_batch_corrected.index.name = 'tx'
ambiguous_counts_batch_corrected.index.name = 'tx'



# Check which transcript IDs exist
exists_mask = adata_isoform.var_names.isin(em_batch_corrected_counts.index)

if not exists_mask.all():
    missing = adata_isoform.var_names[~exists_mask]
    print(f"Warning: {len(missing)} transcript IDs not found in batch_corrected_counts")
    print(f"Missing IDs: {missing.tolist()}")

# Align using reindex (fills missing with NaN by default)
aligned_counts_em = em_batch_corrected_counts.reindex(adata_isoform.var_names)
aligned_counts_unique = unique_counts_batch_corrected.reindex(adata_isoform.var_names)
aligned_counts_ambiguous = ambiguous_counts_batch_corrected.reindex(adata_isoform.var_names)

# Option A: Fill missing values with 0
aligned_counts_em = aligned_counts_em.fillna(0)
aligned_counts_unique = aligned_counts_unique.fillna(0)
aligned_counts_ambiguous = aligned_counts_ambiguous.fillna(0)

# Update the layer
adata_isoform.layers['em_counts'] =  aligned_counts_em.values.T
adata_isoform.layers['unique_counts'] =  aligned_counts_unique.values.T
adata_isoform.layers['ambiguous_counts'] =  aligned_counts_ambiguous.values.T


import numpy as np
em_lib_sizes = np.sum(adata_isoform.layers["em_counts"], axis=1)
em_lib_sizes_expanded = em_lib_sizes[:, np.newaxis]
em_lib_sizes_expanded[em_lib_sizes_expanded == 0] = 1  # Avoid division by zero

adata_isoform.layers["em_cpm"] = (adata_isoform.layers["em_counts"] / em_lib_sizes_expanded) * 1e6
adata_isoform.layers["unique_cpm"] = (adata_isoform.layers["unique_counts"] / em_lib_sizes_expanded) * 1e6
adata_isoform.layers["ambiguous_cpm"] = (adata_isoform.layers["ambiguous_counts"] / em_lib_sizes_expanded) * 1e6

### Add annotation

For interpretation of results we add functional gene annotations from an additional file to the adata.var

In [ ]:
# Load functional annotation data
annotation_file = f"reference/ADAPT_liftoff/Phureja_v4-v6.1_translations.xlsx"
df_annotation = pd.read_excel(annotation_file)
df_annotation = df_annotation.iloc[:, 1:3]
df_annotation.columns = ['gene_id_unitato', 'functional_annotation']

# Remove version suffix from gene IDs (e.g., "GENE.1" -> "GENE")
adata_isoform.var['gene_id_unitato'] = adata_isoform.var['gene_id'].str.rsplit('.', n=1).str[0]
adata_isoform.var['original_index'] = adata_isoform.var.index

# Merge annotations
merged_data = pd.merge(
    adata_isoform.var,
    df_annotation,
    on='gene_id_unitato',
    how='left'
)

# Remove duplicates and restore index
merged_data = merged_data.drop_duplicates(subset=['original_index', 'gene_id'], keep='first')
merged_data.index = merged_data['original_index']
merged_data = merged_data.drop(columns=['original_index'])

# Update AnnData object
adata_isoform.var = merged_data

Let's verify whether Bambu detected any previously unannotated transcripts in our dataset.

In [ ]:
adata_isoform.var[adata_isoform.var['gene_id'].str.startswith("Bambu")].head()

In [ ]:
# print specific gene info
adata_isoform.var[adata_isoform.var['gene_id'].str.contains("Soltu.DM.11G023050")]

Now we will aggregate transcript-level counts to gene-level counts for allele-specific expression analysis. Unique counts are just summed. For ambgious counts we take the mean, under the asseumption that multimapping between transcripts of the same gene to other genes is similar.

In [ ]:
adata_gene = poly.aggregate_transcripts_to_genes(adata_isoform)

## 2) Gene-level analysis

#### Filter low expressed genes


We filter out group of syntenic genes with low expression. Each group need to have at least 20 unique reads mapped per replicate. With `library_size_dependent=True` you can also filter based on libaray size.

#### Calculate allelic ratios and multimapping ratios

After filtering for alleles of equal length, several of the highly biased genes are removed.

Another potential source of bias arises from unequal transcript numbers per gene. For example, a novel transcript may have been identified on only one haplotype. Therefore, we will filter to include only genes where all alleles have the same number of transcripts.

## B) Trans: Testing for differences in allelic expression between conditions

Now we only filter of syntenic group, as multimapping and difference in transcript length should not bias the results when comparing conditions

In [ ]:
adata_gene_filtered = adata_gene.copy()
adata_gene_filtered = poly.filter_low_expressed_genes(
    adata_gene_filtered,
    min_expression=100.0,  # 20 counts
    mode= 'all',
    #library_size_dependent=True, # this filters min_expression based on CPM of unique counts
    verbose=True)

adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'unique_counts')
adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'em_counts')
adata_isoform_filtered = poly.calculate_multi_ratios(adata_gene_filtered, 'unique_counts', 'ambiguous_counts')

In [ ]:
# Filter to only include synteny category "1hap1_1hap2_1hap3_1hap4_s" 
mask = adata_gene_filtered.var["synteny_category"].str.contains("[01]hap1_[01]hap2_[01]hap3_[01]hap4_s", na=False)

adata_gene_syntelogs = adata_gene_filtered[:,mask].copy()

In [ ]:
# Run allelic ratio test
results_df_trans = poly.test_allelic_ratios_between_conditions(adata_gene_syntelogs)

# Get top differential syntelogs
top_results_trans = poly.get_top_differential_syntelogs(results_df_trans, n=50, sort_by='ratio_difference', fdr_threshold=0.05)

# Plot the results
fig = poly.plot_top_differential_syntelogs(top_results_trans, 
                                           n = 20, 
                                           ratio_difference_threshold=0.1, 
                                           sort_by='ratio_difference', 
                                           plot_type='cpm',
                                           sig_threshold=0.05, figsize=(50, 30))

In [ ]:
results_df_trans.sort_values(by='Synt_id', ascending=False, inplace=True)
fig = poly.plot_top_differential_syntelogs(results_df_trans[results_df_trans['gene_id'].str.contains('Soltu.DM.12')], 
                                           n = 40, 
                                           ratio_difference_threshold=0.1, 
                                           sort_by=None, 
                                           plot_type='cpm',
                                           sig_threshold=0.05, figsize=(20, 30))

In [ ]:
results_df_trans[results_df_trans['gene_id'].str.contains('Soltu.DM.12')].sort_values(by='Synt_id', ascending=False).to_csv("LR_DESIREE_PAPER/ANALYSIS/notebooks/Polyase_analysis/DIU_potato_polyase_desiree_batch_corr_Soltu.DM.12.csv", sep="\t", index=False)

In [ ]:
results_df_trans[results_df_trans['gene_id'].str.contains('Soltu.DM.12')].sort_values(by='Synt_id', ascending=False).to_csv("LR_DESIREE_PAPER/ANALYSIS/notebooks/Polyase_analysis/DIU_potato_polyase_desiree_batch_corr_Soltu.DM.12.csv", sep="\t", index=False)

In [ ]:
# # Plot the results
# fig = poly.plot_top_differential_syntelogs(top_results_trans[top_results_trans["gene_id"].str.contains("Soltu.DM.01G044650")], 
#                                            n = 1, figsize=(12,4), 
#                                            ratio_difference_threshold=0.1, 
#                                            sort_by='ratio_difference', 
#                                            plot_type='ratios',
#                                            sig_threshold=0.05)

In [ ]:
# # split the last prefix from transcript_id
# grouped_results['gene_id'] = grouped_results['gene_id'].apply(lambda x: x.rsplit('.', 1)[0] if isinstance(x, str) else x)
#     # Print summary
# significant_results = grouped_results[(grouped_results['FDR'] < 0.05) & (grouped_results['ratio_difference'] > 0.1)]

# unsignificant_results = grouped_results[(grouped_results['FDR'] >= 0.05) | (grouped_results['ratio_difference'] <= 0.1)]

# # Filter the syntelog results with balanced expression for plotting
# mask = unsignificant_results.index
# balanced_trans_results = results_df_trans[results_df_trans['Synt_id'].isin(unsignificant_results.index)]
# fig = poly.plot_top_differential_syntelogs(balanced_trans_results, 
#                                            n = 20)



#### Plot genes we selected for promotor testig 

In [ ]:
adata_gene.var[adata_gene.var['gene_id'].str.contains("Soltu.DM.08G022800")]
results_df_trans[results_df_trans['gene_id'].str.contains('Soltu.DM.08G022800')]

In [ ]:
gene_id = ["Soltu.DM.05G007650", "Soltu.DM.08G022800", "Soltu.DM.12G011670", "Soltu.DM.01G002150"]

# filter results for specific synt id
trans_results_filtered_synt = results_df_trans[results_df_trans['gene_id'].str.contains('|'.join(gene_id))]
ax1 = poly.plot_top_differential_syntelogs(trans_results_filtered_synt, n = 20, figsize = (16, 3*1),sort_by='ratio_difference')

ax2 = poly.plot_top_differential_syntelogs(trans_results_filtered_synt, n = 20, figsize = (16
                                                                                        , 3*1),sort_by='ratio_difference', plot_type='cpm' )

## 3) Isoform level analysis

In the previous steps we tested for differences in in allelic expression on gene level, no we are looking at the isoform level

##### Filtering out genes with low expression

In [ ]:
adata_isoform_filtered = poly.filter_low_expressed_genes(
    adata_isoform,
    min_expression=100.0,  # 20 counts
    mode= 'all',
    group_col = 'gene_id',
    verbose=True,
    layer='em_counts'
    )

In [ ]:
adata_isoform_filtered = poly.calculate_allelic_ratios(adata_isoform_filtered, 'unique_counts' )
adata_isoform_filtered = poly.calculate_multi_ratios(adata_isoform_filtered, 'unique_counts', 'ambiguous_counts')
adata_isoform_filtered = poly.calculate_per_allele_ratios(adata_isoform_filtered, unique_layer='unique_counts', multi_layer='ambiguous_counts')

### C) Testing for differential isoform usage between conditions

Now we are testing for differential isofrom usage (DIU). Are there different isoforms expressed in tuber than in leaves?

In [ ]:
# select only the genes with equal lengths, low multimapping ratio and synteny category "1hap1_1hap2_1hap3_1hap4_s"
mask =  (adata_isoform_filtered.var["tx_multimapping_ratio_per_allele_weighted_average"] <= 1)
adata_isoform_multimapping_filter = adata_isoform_filtered[:,mask].copy()

# Test for differential allelic ratios withing conditions
DIU_results, DIU_plotting_result = poly.test_isoform_DIU_between_conditions(adata_isoform_multimapping_filter, 
                                                                       layer="em_counts", 
                                                                       inplace=True)

In [ ]:
# export DIU results
# add log2FC and log2FC absolute valu
#- also export mean CPMs to the other table! for saving! and calculate log2FC based on this
DIU_results['log2FC'] = np.log2((DIU_results['ratios_control_mean']) / (DIU_results['ratios_heat_mean']))
DIU_results.to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/DIU_results.csv", sep=",")

## check strange result





For plotting of differential isoform usage we also need to add the gene structure from the gtf files.

In [ ]:
ensembl_gtf_path = "reference/Desiree_v1/De_v1.unitato_liftoff_haplotap_gffread.gtf"
annotation = RNApy.read_ensembl_gtf(ensembl_gtf_path)

# add the bambu annotations to the annotation dataframe
# with bambu
bambu = "nf-core-plantlongrnaseq/output_desiree_liftoff_phased/gff/combined.fixed.gtf"
bambu_annotation = RNApy.read_ensembl_gtf(bambu)
bambu_annotation = bambu_annotation.filter(pl.col("transcript_id").str.contains("Bambu"))


annotation.extend(bambu_annotation)
annotation_df = annotation.to_pandas()
annotation_df['gene_id_unitato'] = annotation_df['gene_id'].str.rsplit('.', n=1).str[0]

# back to polar
annotation_pl = pl.from_pandas(annotation_df)

## Plot overall DIU results

### Export DIU images

#### SPA1

In [ ]:
figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.startswith("Soltu.DM.12G029670")],
    annotation_df=annotation_pl,
    fdr_threshold=1,
    ratio_difference_threshold=0
)
# save as svg
#figures[1].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU_plot_Soltu.DM.12G029670.svg")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7, 5))

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.12G029670.Hap2",
    include=["Soltu.DM.12G029670.1.Hap2", "Soltu.DM.12G029670.2.Hap2"],
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    ax=axes[0],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.8,
)

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.12G029670.Hap2",
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    aggregate='sum',
    ax=axes[1],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.5,
)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/SPA3_expression.svg", format="svg")
plt.show()

## HSP40

In [ ]:
figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.startswith("Soltu.DM.03G037350")],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.1
)
# save as svg
figures[1].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU_plot_Soltu.DM.03G037350.svg")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7, 5))

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.03G037350.Hap4",
    include=["Soltu.DM.03G037350.1.Hap4", "Soltu.DM.03G037350.2.Hap4"],
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    ax=axes[0],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=1,
)

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.03G037350.Hap4",
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    aggregate='sum',
    ax=axes[1],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=1,
)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/HSP40_hap4_expression.svg", format="svg")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7, 5))

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.03G037350.Hap2",
    include=["Soltu.DM.03G037350.1.Hap2", "Soltu.DM.03G037350.2.Hap2"],
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    ax=axes[0],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=1,
)

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.03G037350.Hap2",
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    aggregate='sum',
    ax=axes[1],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=1,
)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/HSP40_expression.svg", format="svg")
plt.show()

###

## HSFA6b


In [ ]:
figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.startswith("Soltu.DM.09G020350")],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.1
)
figures[0].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/DIU_plot_HSFA6b.svg")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(6, 5))

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.09G020350.Hap1",
    include=["Soltu.DM.09G020350.1.Hap1", "Soltu.DM.09G020350.3.Hap1"],
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    ax=axes[0],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=1,
)

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.09G020350.Hap1",
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    aggregate='sum',
    ax=axes[1],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=1,
)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/HSFA6b_expression.svg", format="svg")
plt.show()

### Alpha beta anhydrase



In [ ]:
figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.startswith("Soltu.DM.05G008530")],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.0
)
figures[0].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU_plot_anhydrase_Soltu.DM.05G008530.svg")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(6, 5
                                        ))

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.05G008530.Hap2",
    #include=["Soltu.DM.12G023350.2.Hap2", "Soltu.DM.12G023350.3.Hap2"],
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    ax=axes[0],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.8,
)

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.05G008530.Hap2",
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    aggregate='sum',
    ax=axes[1],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.5,
)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/anhydrase_expression.svg", format="svg")
plt.show()

### COP1

In [ ]:

figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.contains("Soltu.DM.12G023350", na=False)],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.1
)
figures[0].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/DIU_plot_COP1_Soltu.DM.12G023350.Hap2.svg")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(6, 5))

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.12G023350.Hap2",
    include=["Soltu.DM.12G023350.2.Hap2", "Soltu.DM.12G023350.3.Hap2"],
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    ax=axes[0],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.8,
)

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.12G023350.Hap2",
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    aggregate='sum',
    ax=axes[1],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.5,
)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/COP1_expression.svg", format="svg")
plt.show()

## JAZ3

In [ ]:
figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.contains("Soltu.DM.01G000760", na=False)],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.1
)
# save as svg
figures[0].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU_plot_JAZ_Soltu.DM.01G000760.svg")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(6, 5))

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.01G000760.Hap1",
    include=["Soltu.DM.01G000760.1.Hap1", "Soltu.DM.01G000760.2.Hap1"],
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    ax=axes[0],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.8,
)

poly.plot_expression(
    adata_isoform,
    query="Soltu.DM.01G000760.Hap1",
    id_type='gene_id',
    layer='em_cpm',
    group_key='condition',
    aggregate='sum',
    ax=axes[1],
    palette={'control': "#a0a8af", 'heat': "#efa31f"},
    jitter=0.2,
    alpha=0.5,
)

plt.tight_layout()
# save as svg
plt.savefig("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/JAZ_expression.svg", format="svg")
plt.show()

In [ ]:
DIU_results_orig = DIU_results.copy()
DIU_results = DIU_results_orig.copy()
DIU_results["ratio_difference"] = DIU_results['ratios_heat_mean'] - DIU_results['ratios_control_mean']
# filter for top isoform per gene
DIU_results['ratio_difference_abs'] = DIU_results['ratio_difference'].abs()
# take for each gene the isoform with the highest expression (mean of heat and control) and lowest FDR

DIU_results_top_isoform = DIU_results.sort_values('FDR', ascending=True).groupby('gene_id').head(1)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# ---------------------------------------------------------------
# Thresholds
# ---------------------------------------------------------------
fdr_thresh = 0.05
dpsi_thresh = 0.1

# ---------------------------------------------------------------
# Build working dataframe first, then derive columns from it
# ---------------------------------------------------------------
DIU_results = DIU_results_top_isoform.copy()
DIU_results['log_fdr'] = -np.log10(DIU_results['FDR'])

def classify(row):
    sig = row['log_fdr'] > -np.log10(fdr_thresh)
    if sig and row['ratio_difference'] > dpsi_thresh:
        return 'UP'
    if sig and row['ratio_difference'] < -dpsi_thresh:
        return 'DOWN'
    return 'NS'

DIU_results['category'] = DIU_results.apply(classify, axis=1)

# EnhancedVolcano-style palette
palette = {'UP': '#d62728', 'DOWN': '#d62728', 'NS': '#8c8c8c'}
# (EnhancedVolcano typically colors both UP and DOWN significant hits red —
#  "p-value and log2FC" — change DOWN to a different color here if you want
#  UP/DOWN split instead of the single "significant" category.)

# ---------------------------------------------------------------
# Gene ID -> display name mapping
# ---------------------------------------------------------------
gene_name_map = {
    'Soltu.DM.05G008530.Hap1': 'CA.hap1',
    'Soltu.DM.05G008530.Hap2': 'CA.hap2',
    'Soltu.DM.05G008530.Hap3': 'CA.hap3',
    'Soltu.DM.05G008530.Hap4': 'CA.hap4',
    'Soltu.DM.03G037350.Hap1': 'HSP40.hap1',
    'Soltu.DM.03G037350.Hap2': 'HSP40.hap2',
    'Soltu.DM.03G037350.Hap3': 'HSP40.hap3',
    'Soltu.DM.03G037350.Hap4': 'HSP40.hap4',
    'Soltu.DM.12G029670.Hap1': 'SPA3.hap1',
    'Soltu.DM.12G029670.Hap2': 'SPA3.hap2',
    'Soltu.DM.12G029670.Hap3': 'SPA3.hap3',
    'Soltu.DM.12G029670.Hap4': 'SPA3.hap4',
    "Soltu.DM.01G000760.Hap1": 'JAZ.hap1',
    "Soltu.DM.01G000760.Hap2": 'JAZ.hap2',
    "Soltu.DM.01G000760.Hap3": 'JAZ.hap3',
    "Soltu.DM.01G000760.Hap4": 'JAZ.hap4',
    'Soltu.DM.12G023350.Hap1': 'COP1.hap1',
    'Soltu.DM.12G023350.Hap2': 'COP1.hap2',
    'Soltu.DM.12G023350.Hap3': 'COP1.hap3',
    'Soltu.DM.12G023350.Hap4': 'COP1.hap4',
    'Soltu.DM.09G020350.Hap1': 'HSFA6.hap1',
    'Soltu.DM.09G020350.Hap2': 'HSFA6.hap2',
    'Soltu.DM.09G020350.Hap3': 'HSFA6.hap3',
    'Soltu.DM.09G020350.Hap4': 'HSFA6.hap4',
}

highlight = set(gene_name_map.keys())
highlight_mask = (
    DIU_results['gene_id'].str.startswith(tuple(highlight)) &
    (DIU_results['category'] != 'NS')
)

# ---------------------------------------------------------------
# Base styling
# ---------------------------------------------------------------
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'grid.color': '#ececec',
    'grid.linewidth': 0.5,
})

fig, ax = plt.subplots(figsize=(4.5, 4.5))

ax.set_axisbelow(True)
ax.grid(True, linestyle='-', linewidth=0.5, color='#ececec')
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(0.8)

# ---------------------------------------------------------------
# Scatter — NS behind, significant points on top
# ---------------------------------------------------------------
for cat in ['NS', 'DOWN', 'UP']:
    grp = DIU_results[DIU_results['category'] == cat]
    if grp.empty:
        continue
    ax.scatter(grp['ratio_difference'], grp['log_fdr'],
               color=palette[cat], alpha=0.75, s=16,
               linewidths=0, zorder=2)

# threshold lines: dashed black, thin
ax.axhline(-np.log10(fdr_thresh), color='black', linestyle=(0, (6, 4)),
            linewidth=1, zorder=1)
ax.axvline(dpsi_thresh,  color='black', linestyle=(0, (6, 4)), linewidth=0.8, zorder=1)
ax.axvline(-dpsi_thresh, color='black', linestyle=(0, (6, 4)), linewidth=0.8, zorder=1)

# ---------------------------------------------------------------
# Highlighted genes of interest
# ---------------------------------------------------------------
for _, row in DIU_results[highlight_mask].iterrows():
    ax.scatter(row['ratio_difference'], row['log_fdr'],
               color=palette[row['category']], s=45,
               edgecolors='black', linewidths=0.6, zorder=5)

    matched_key = next((k for k in gene_name_map if row['gene_id'].startswith(k)), None)
    label = gene_name_map.get(matched_key, row['transcript_id'])

    ax.annotate(label, xy=(row['ratio_difference'], row['log_fdr']),
                xytext=(4, 3), textcoords='offset points',
                fontsize=10, fontweight='normal', color='black', zorder=6)

# ---------------------------------------------------------------
# Axis labels
# ---------------------------------------------------------------
ax.set_xlabel(r'$Log_2$ fold change', fontsize=11)
ax.set_ylabel(r'$-Log_{10}$ FDR', fontsize=11)
ax.tick_params(labelsize=11)

# ---------------------------------------------------------------
# Title / subtitle block (top-left, like EnhancedVolcano)
# ---------------------------------------------------------------
fig.text(0.02, 0.98, 'voom lm fit', fontsize=13, fontweight='bold', ha='left', va='top')
fig.text(0.02, 0.935, 'EnhancedVolcano', fontsize=10, fontstyle='italic', ha='left', va='top')

# ---------------------------------------------------------------
# Horizontal dot-legend row (mimics the 4-category legend row)
# ---------------------------------------------------------------
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#8c8c8c',
               markersize=6, label='NS'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728',
               markersize=6, label='p-value and log$_2$ FC'),
]
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.89),
           ncol=2, frameon=False, fontsize=8, handletextpad=0.3, columnspacing=1.2)

plt.subplots_adjust(top=0.8)
plt.savefig(
    "LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/DIU_volcano_plot_top_isoform.svg",
    format='svg', dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# figures = poly.plot_differential_isoform_usage(
#     results_df=DIU_plotting_result,#[DIU_plotting_result['gene_id'].str.startswith("Soltu.DM.03G037350")],
#     annotation_df=annotation_pl,
#     fdr_threshold=0.05,
#     ratio_difference_threshold=0.1
# )

In [ ]:
DIU_plotting_result
# filter based on if at least one transcript id per gene_id contina "XLOC" or "Bambu"
genes_w_novel_isoforms = DIU_plotting_result.groupby('gene_id').filter(lambda x: x['gene_id'].str.contains("XLOC|Bambu").any())

# filter the plotting result for only the genes with novel isoforms
DIU_plotting_result_novel_isoforms = DIU_plotting_result[DIU_plotting_result['gene_id'].isin(genes_w_novel_isoforms['gene_id'])]


In [ ]:
DIU_results_top_isoform[DIU_results_top_isoform['gene_id'].str.contains("Soltu.DM.12G029670")]

In [ ]:
# # get a list of the significant genes
# significant_genes = DIU_results[(DIU_results['FDR'] < 0.05) & (DIU_results['ratio_difference'] > 0.1)]['gene_id'].str.rsplit('.', n=1).str[0].unique().tolist()
# unsignificant_genes = DIU_results['gene_id'].str.rsplit('.', n=1).str[0].unique().tolist()
# # save the significant genes to a file
# pd.DataFrame(significant_genes, columns=['gene_id']).to_csv("LR_DESIREE_PAPER/DIU_results/significant_genes_0.05_0.1_isoform_DIU_between_conditions.csv", index=False)
# pd.DataFrame(unsignificant_genes, columns=['gene_id']).to_csv("LR_DESIREE_PAPER/DIU_results/unsignificant_genes_0.05_0.1_isoform_DIU_between_conditions.csv", index=False)

In [ ]:
# figures = poly.plot_differential_isoform_usage(
#     results_df=DIU_plotting_result, #[DIU_plotting_result['functional_annotation'].str.contains("heat", na=False)],
#     annotation_df=annotation_pl,
#     fdr_threshold=0.05,
#     ratio_difference_threshold=0.5
# )

In [ ]:
figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['functional_annotation'].str.contains("shock", na=False)],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.1)

In [ ]:

figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.startswith("Soltu.DM.03G017140")],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.1
)

In [ ]:
# Soltu.DM.09G020350.Hap3

figures = poly.plot_differential_isoform_usage(
    results_df=DIU_plotting_result[DIU_plotting_result['gene_id'].str.startswith("Soltu.DM.09G020350")],
    annotation_df=annotation_pl,
    fdr_threshold=0.05,
    ratio_difference_threshold=0.1
)

## HSF

### D) Testing for differential isofrom strucuture between haplotypes


Now we will test if there are any genes that have a different isoform expressed on the different haplotypes. This can be caused for instance by mutations in splice sites.

In [ ]:
#First add exon structure information to your AnnData object
gtf_file = bambu

# Add structure information (this can take some time)
poly.add_structure_from_gtf(adata_isoform_filtered, 
                            gtf_file, 
                            inplace=True, 
                            verbose=True)

## Isofrom strucutre haplotypte HEAT

In [ ]:

adata_isoform_filtered = poly.filter_low_expressed_genes(
    adata_isoform[(adata_isoform.obs['condition'] == 'heat'), :],
    min_expression=100.0,  # 20 counts
    mode= 'all',
    group_col = 'gene_id',
    verbose=True)

adata_isoform_filtered = poly.calculate_allelic_ratios(adata_isoform_filtered, 'unique_counts' )
adata_isoform_filtered = poly.calculate_multi_ratios(adata_isoform_filtered, 'unique_counts', 'ambiguous_counts')
adata_isoform_filtered = poly.calculate_per_allele_ratios(adata_isoform_filtered, unique_layer='unique_counts', multi_layer='ambiguous_counts')

poly.add_structure_from_gtf(adata_isoform_filtered, 
                            gtf_file, 
                            inplace=True, 
                            verbose=True)

mask =(adata_isoform_filtered.var["tx_multimapping_ratio_per_allele_weighted_average"] < 1) & \
      (adata_isoform_filtered.var["synteny_category"].str.contains("1hap1_1hap2_1hap3_1hap4_s|0hap1_1hap2_1hap3_1hap4_s|1hap1_0hap2_1hap3_1hap4_s|1hap1_1hap2_0hap3_1hap4_s|1hap1_1hap2_1hap3_0hap4_s|0hap1_0hap2_1hap3_1hap4_s|1hap1_1hap2_0hap3_0hap4_s|0hap1_1hap2_1hap3_0hap4_s|1hap1_0hap2_0hap3_1hap4_s", na=False))

#mask = adata_isoform_filtered.var["Synt_id"] == "Synt_id_59067"
adata_isoform_multimapping_filtered = adata_isoform_filtered[:,mask].copy()


#  Run structure-based DIU analysis
results_DIU_allele_HEAT, results_DIU_plotting_HEAT = poly.test_differential_isoform_structure(
    adata_isoform_multimapping_filtered, 
    layer="unique_counts",
    test_condition="all",  # or "all" to use all conditions
    min_similarity_for_matching=0.90,
    inplace=True
)
results_DIU_allele_HEAT.to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/HEAT_Different_isoform_structure_results.csv")

## Isofrom strucutre CONTROL


In [ ]:

adata_isoform_filtered = poly.filter_low_expressed_genes(
    adata_isoform[(adata_isoform.obs['condition'] == 'control'), :],
    min_expression=100.0,  # 20 counts
    mode= 'all',
    group_col = 'gene_id',
    verbose=True)

adata_isoform_filtered = poly.calculate_allelic_ratios(adata_isoform_filtered, 'unique_counts' )
adata_isoform_filtered = poly.calculate_multi_ratios(adata_isoform_filtered, 'unique_counts', 'ambiguous_counts')
adata_isoform_filtered = poly.calculate_per_allele_ratios(adata_isoform_filtered, unique_layer='unique_counts', multi_layer='ambiguous_counts')

poly.add_structure_from_gtf(adata_isoform_filtered, 
                            gtf_file, 
                            inplace=True, 
                            verbose=True)

mask =(adata_isoform_filtered.var["tx_multimapping_ratio_per_allele_weighted_average"] < 1) & \
      (adata_isoform_filtered.var["synteny_category"].str.contains("1hap1_1hap2_1hap3_1hap4_s|0hap1_1hap2_1hap3_1hap4_s|1hap1_0hap2_1hap3_1hap4_s|1hap1_1hap2_0hap3_1hap4_s|1hap1_1hap2_1hap3_0hap4_s|0hap1_0hap2_1hap3_1hap4_s|1hap1_1hap2_0hap3_0hap4_s|0hap1_1hap2_1hap3_0hap4_s|1hap1_0hap2_0hap3_1hap4_s", na=False))

#mask = adata_isoform_filtered.var["Synt_id"] == "Synt_id_59067"
adata_isoform_multimapping_filtered = adata_isoform_filtered[:,mask].copy()


#  Run structure-based DIU analysis
results_DIU_allele_CONTROL, results_DIU_plotting_CONTROL = poly.test_differential_isoform_structure(
    adata_isoform_multimapping_filtered, 
    layer="em_counts",
    test_condition="all",  # or "all" to use all conditions
    min_similarity_for_matching=0.95,
    inplace=True
)
results_DIU_allele_CONTROL.to_csv("LR_DESIREE_PAPER/ANALYSIS/final_tables_paper/CONTROL_Different_isoform_structure_results.csv")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------
# Thresholds
# ---------------------------------------------------------------
fdr_thresh = 0.05
dpsi_thresh = 0.1

# ---------------------------------------------------------------
# Gene ID -> display name mapping (gene-level, no .HapN suffix)
# ---------------------------------------------------------------
gene_name_map = {
    'Soltu.DM.02G008950': 'ABH',
}
highlight = set(gene_name_map.keys())

# ---------------------------------------------------------------
# Collapse to one row per Synt_id: pick the most significant
# (lowest p_value) pairwise haplotype comparison per gene
# ---------------------------------------------------------------
DIU_hap_results = results_DIU_allele_CONTROL.copy()

idx_best = (
    DIU_hap_results
    .dropna(subset=['p_value'])
    .groupby('Synt_id')['p_value']
    .idxmin()
)
DIU_per_gene = DIU_hap_results.loc[idx_best].copy()

# ---- recompute FDR across these one-per-gene representative p-values ----
DIU_per_gene['FDR'] = multipletests(DIU_per_gene['p_value'], method='fdr_bh')[1]
DIU_per_gene['log_fdr'] = -np.log10(DIU_per_gene['FDR'].clip(lower=1e-300))

# ---- derive gene_id from reference_transcript ----
DIU_per_gene['gene_id'] = (
    DIU_per_gene['reference_transcript']
    .str.replace(r'\.Hap\d+$', '', regex=True)
    .str.replace(r'\.\d+$', '', regex=True)
)

# ---- classify ----
def classify(row):
    sig = row['log_fdr'] > -np.log10(fdr_thresh)
    if sig and row['ratio_difference'] > dpsi_thresh:
        return 'UP'
    if sig and row['ratio_difference'] < -dpsi_thresh:
        return 'DOWN'
    return 'NS'

DIU_per_gene['category'] = DIU_per_gene.apply(classify, axis=1)

palette = {'UP': '#d62728', 'DOWN': '#d62728', 'NS': '#8c8c8c'}

highlight_mask = (
    DIU_per_gene['gene_id'].str.startswith(tuple(highlight)) &
    (DIU_per_gene['category'] != 'NS')
)

# ---------------------------------------------------------------
# Styling (EnhancedVolcano-like)
# ---------------------------------------------------------------
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.8,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'grid.color': '#ececec',
    'grid.linewidth': 0.5,
})

fig, ax = plt.subplots(figsize=(3.5, 4.5))
ax.set_axisbelow(True)
ax.grid(True, linestyle='-', linewidth=0.5, color='#ececec')
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(0.8)

for cat in ['NS', 'DOWN', 'UP']:
    grp = DIU_per_gene[DIU_per_gene['category'] == cat]
    if grp.empty:
        continue
    ax.scatter(grp['ratio_difference'], grp['log_fdr'],
               color=palette[cat], alpha=0.75, s=16,
               linewidths=0, zorder=2)

ax.axhline(-np.log10(fdr_thresh), color='black', linestyle=(0, (6, 4)),
            linewidth=1, zorder=1)
ax.axvline(dpsi_thresh,  color='black', linestyle=(0, (6, 4)), linewidth=1, zorder=1)
#ax.axvline(-dpsi_thresh, color='black', linestyle=(0, (6, 4)), linewidth=1, zorder=1)

for _, row in DIU_per_gene[highlight_mask].iterrows():
    ax.scatter(row['ratio_difference'], row['log_fdr'],
               color=palette[row['category']], s=45,
               edgecolors='black', linewidths=0.6, zorder=5)
    matched_key = next((k for k in gene_name_map if row['gene_id'].startswith(k)), None)
    label = gene_name_map.get(matched_key, row['reference_transcript'])
    ax.annotate(f"{label} ({row['reference_haplotype']} vs {row['comparison_haplotype']})",
                xy=(row['ratio_difference'], row['log_fdr']),
                xytext=(4, 3), textcoords='offset points',
                fontsize=11, color='black', zorder=6)

ax.set_xlabel(r'$\Delta$ isoform ratio', fontsize=11)
# set x-limits to focus on the range of interest
xlim_min = 0
xlim_max = DIU_per_gene['ratio_difference'].abs().max() + 0.05
ax.set_xlim(xlim_min, xlim_max)
ax.set_ylabel(r'$-Log_{10}$ FDR', fontsize=11)
ax.tick_params(labelsize=11)

fig.text(0.02, 0.98, 'Isoform usage: haplotype comparison',
          fontsize=12, fontweight='bold', ha='left', va='top')
fig.text(0.02, 0.935, 'one point per Synt_id (best pairwise comparison)',
          fontsize=9, fontstyle='italic', ha='left', va='top')

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#8c8c8c',
               markersize=6, label='NS'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728',
               markersize=6, label='p-value and $\\Delta$ ratio'),
]
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.89),
           ncol=2, frameon=False, fontsize=11, handletextpad=0.3, columnspacing=1.2)

plt.subplots_adjust(top=0.8)
plt.savefig(
    "LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/DIU/DIU_volcano_per_synt_id.svg",
    format='svg', dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
figures = poly.plot_allele_specific_isoform_structure(
    results_df=results_DIU_plotting_CONTROL,
    annotation_df=annotation_pl,
    ratio_difference_threshold=0.5
)

## Plot alpha-beta hydroxylase family protein

In [ ]:

figures = poly.plot_allele_specific_isoform_structure(
    results_df=results_DIU_plotting_HEAT[results_DIU_plotting_HEAT['gene_id'].str.startswith("Soltu.DM.02G008950")],
    annotation_df=annotation_pl,
    ratio_difference_threshold=0.1
)

# save as svg
figures[0].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/allele_specific_isoform_structure_Soltu.DM.02G008950.svg")

## Plot APG3

In [ ]:
figures = poly.plot_allele_specific_isoform_structure(
    results_df=results_DIU_plotting_HEAT[results_DIU_plotting_HEAT['gene_id'].str.startswith("Soltu.DM.06G012460")],
    annotation_df=annotation_pl,
    ratio_difference_threshold=0.1
)

# save as svg
figures[0].write_image("LR_DESIREE_PAPER/ANALYSIS/final_plots_svg/allele_specific_isoform_structure_Soltu.DM.06G012460.svg")